# Longitudinal imaging features for dementia risk (public data)

Compare age and MRI-derived volumes at the prediction visit with the same features plus annualized changes since the prior visit. Evaluate regularized logistic regression on subject-level out-of-fold probabilities. Use the official OASIS-2 demographic workbook. Require two visits with CDR zero followed by at least one subsequent assessment. The target is any subsequent CDR above zero during available follow-up. The cohort contains 36 eligible participants and only four events. Use four-fold stratification and treat estimates as exploratory. Features are derived from MRI summaries.

In [ ]:
from pathlib import Path
import subprocess, sys
root = Path.cwd() / 'longitudinal-imaging-landmark-real-data-run'
root.mkdir(exist_ok=True)
subprocess.run([sys.executable,'-m','pip','install',*['numpy==2.2.6', 'scipy==1.15.3', 'scikit-learn==1.7.2', 'joblib==1.6.0', 'openpyxl==3.1.5']],check=True)

In [ ]:
(root / 'prepare.py').write_text('import os\nfrom pathlib import Path\nfrom urllib.request import urlretrieve\n\ndata = Path(os.environ.get("PORTFOLIO_DATA", Path(__file__).resolve().parent / "data"))\ndata.mkdir(parents=True, exist_ok=True)\ntarget = data / "oasis2.xlsx"\nif not target.exists():\n    temp = target.with_suffix(".part")\n    urlretrieve(\n        "https://sites.wustl.edu/oasisbrains/files/2024/03/oasis_longitudinal_demographics-8d83e569fa2e2d30.xlsx",\n        temp,\n    )\n    temp.replace(target)\n',encoding='utf8')
subprocess.run([sys.executable,'prepare.py'],cwd=root,check=True)

In [ ]:
(root / 'train.py').write_text('import json\nimport os\nfrom pathlib import Path\n\nimport joblib\nimport numpy as np\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.metrics import (\n    average_precision_score,\n    brier_score_loss,\n    roc_auc_score,\n)\nfrom sklearn.pipeline import make_pipeline\nfrom sklearn.preprocessing import StandardScaler\n\nROOT = Path(__file__).resolve().parent\nDATA = Path(os.environ.get("PORTFOLIO_DATA", ROOT / "data"))\nSEEDS = [13, 37, 61]\n\n\ndef save(obj, name):\n    folder = ROOT / "checkpoints"\n    folder.mkdir(exist_ok=True)\n    joblib.dump(obj, folder / (name + ".joblib"), compress=3)\n\n\ndef scores(y, p):\n    return {\n        "auroc": float(roc_auc_score(y, p)),\n        "average_precision": float(average_precision_score(y, p)),\n    }\n\n\nfrom collections import defaultdict\n\nfrom openpyxl import load_workbook\n\n\ndef dataset(seed):\n    w = load_workbook(DATA / "oasis2.xlsx", read_only=True, data_only=True)\n    rows = w.active.values\n    fields = next(rows)\n    subjects = defaultdict(list)\n    for row in rows:\n        rec = dict(zip(fields, row))\n        if rec["Subject ID"] and rec["CDR"] is not None:\n            subjects[rec["Subject ID"]].append(rec)\n    x = []\n    y = []\n    keys = ["nWBV", "eTIV", "ASF"]\n    for visits in subjects.values():\n        visits.sort(key=lambda v: v["MR Delay"])\n        if len(visits) < 3:\n            continue\n        first, second, *future = visits\n        years = (second["MR Delay"] - first["MR Delay"]) / 365.25\n        if first["CDR"] != 0 or second["CDR"] != 0 or years <= 0:\n            continue\n        # Follow-up labels use only visits after the prediction landmark.\n        y.append(int(any(v["CDR"] > 0 for v in future)))\n        x.append(\n            [\n                second["Age"],\n                *[second[k] for k in keys],\n                *[(second[k] - first[k]) / years for k in keys],\n            ]\n        )\n    assert sum(y) >= 4, (len(y), sum(y))\n    return np.array(x), np.array(y)\n\n\nfrom sklearn.impute import SimpleImputer\nfrom sklearn.model_selection import StratifiedKFold\n\n\ndef run(seed):\n    x, y = dataset(seed)\n    results = {}\n    folds = list(\n        StratifiedKFold(n_splits=4, shuffle=True, random_state=seed).split(x, y)\n    )\n    for name, k in [("latest_visit", 4), ("longitudinal", 7)]:\n        p = np.empty(len(y))\n        for i, (train, test) in enumerate(folds):\n            model = make_pipeline(\n                SimpleImputer(),\n                StandardScaler(),\n                LogisticRegression(C=0.08, max_iter=700),\n            ).fit(x[train, :k], y[train])\n            p[test] = model.predict_proba(x[test, :k])[:, 1]\n            save(model, f"{name}-{seed}-{i}")\n        results[name] = {\n            **scores(y, p),\n            "brier": float(brier_score_loss(y, p)),\n            "subjects": len(y),\n            "events": int(y.sum()),\n        }\n    return results\n\n\nif __name__ == "__main__":\n    runs = {s: run(s) for s in SEEDS}\n    first = next(iter(runs.values()))\n    means = {\n        k: {m: float(np.mean([r[k][m] for r in runs.values()])) for m in v}\n        for k, v in first.items()\n    }\n    (ROOT / "results.json").write_text(\n        json.dumps({"seeds": SEEDS, "runs": runs, "means": means}, indent=2)\n    )\n    print(means)\n',encoding='utf8')
subprocess.run([sys.executable,'train.py'],cwd=root,check=True)

In [ ]:
print((root/'results.json').read_text())